# Week 07 · Day 1 — LLM Engineering Foundations
**Transformers, Tokens, Prompting, and Your First Local Model**

**Date:** 2026-09-15 · **Machine:** Dell Latitude 5590, 31 GiB RAM, CPU-only, Ubuntu · **Runtime:** Ollama 0.33.3

## What this notebook covers
This is the first day of the week where "the system" is a model, not just code we
write. By the end you should be able to explain a transformer conceptually, know what
tokens and context windows really are, apply the five core prompting techniques, and
have **two real open-weight models running and verified on your own machine** — no API
key, no cloud bill, no network dependency after download.

**How to read this notebook:** every section follows *question → what we did → what we
found*. The code cells produce the evidence; the markdown tells the story. A reader
who skips every code cell still understands the full picture.

## The framing that matters
> Before you touch a hosted API, you should see with your own eyes that a large
> language model is a **real, ordinary file sitting on disk** that a program loads and
> runs — not a mysterious remote service behind someone else's endpoint.

That is exactly what we did in Part C: two local models pulled to disk, loaded into
RAM, and answering real questions, all without a single hosted API call.

# Part A — Concepts, Made Concrete

## A0. What a Transformer Actually Does (conceptually)

**Question:** what happens when you send a prompt to a transformer-based LLM?

**What we know (no math needed):**
1. A transformer does one repeated operation at inference time: given everything so far,
   predict the most probable **next token**, append it, and repeat. Text is generated
   token by token, not word by word.
2. **Attention** is the mechanism that lets the model weigh which earlier tokens in the
   input matter most for predicting the next one. A pronoun late in a paragraph can
   attend to the noun it refers to several sentences earlier — that is why a
   transformer can track long-range structure at all, and why it beat the previous
   generation (RNNs/LSTMs) that had vanishing memory.
3. The **consequence** of all this: the model has **no persistent memory** of anything
   outside what is literally present in the current request's text. When a conversation
   "forgets" an earlier detail, that is a context-window problem — not a memory bug in
   the model.

**What this means for us:** every prompting technique in Part B works *within* the
current request. The model cannot remember — so the request itself must carry anything
the model needs.

## A1. Tokens — the actual unit the model reads (and will be billed on)

**Question:** what exactly is a token, and why is it not a word?

**What we did:** tokenized 3–4 sentences of our own writing (from a real daily status
update) using `tiktoken`, OpenAI's public tokenizer — the same cl100k_base encoding
used for GPT-4-class models. Seeing the actual boundaries makes "a token" stop being an
abstraction.

In [1]:
import tiktoken

# Sampled from my own daily status update (3-4 sentences)
my_writing = (
    "Completed an in-depth, concept-by-concept walkthrough of the ML Pipeline Practical: "
    "went through the entire pipeline one step at a time and for each step wrote down its "
    "core concept in a line and re-verified its actual numbers from a fresh live run."
)

enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(my_writing)
words = my_writing.split()

print(f"Words:       {len(words)}")
print(f"Tokens:      {len(tokens)}")
print(f"Tokens/word: {len(tokens)/len(words):.2f}")
print(f"Tokens/char: {len(tokens)/len(my_writing):.3f}")
print()
print("Surprising token boundaries:")
interesting = []
for i, chunk in enumerate(enc.decode_tokens_bytes(tokens)):
    d = chunk.decode("utf-8", errors="replace").replace("\n", "\\n")
    # flag multi-char tokens that contain punctuation or are split compounds
    if any(c in d for c in "-.,:;"):
        interesting.append(d)
for d in interesting:
    print(f"  {d!r}")

Words:       42
Tokens:      51
Tokens/word: 1.21
Tokens/char: 0.205

Surprising token boundaries:
  '-depth'
  ','
  '-by'
  '-con'
  ':'
  '-'
  '.'


**What this tells us (Finding A1):**
- 42 words became **51 tokens** — **1.21 tokens per word**, and about **0.2 tokens per
  character**. A token is *not* a word.
- Surprising boundaries observed: the compound `concept-bye-concept` text splits into
  pieces like `[-by][-con][cept]`, and `re-verified` becomes `[re][-][verified]` (the
  hyphen is its own token). Common words (`Pipeline`, `Practical`) stay whole because
  the tokenizer has seen them often.
- **Implication:** "roughly 4 characters or ¾ of a word" is a good mental model, but
  exact counts vary per provider and per language — never estimate pricing or window
  usage from words; count tokens.

**Source:** `part_a/tokenizer_exercise.py` · **Evidence:** `evidence/partA1_tokenizer.txt`

## A2. Context Window — why the model "forgets", and why it isn't forgetting

**Question:** how many turns of a real conversation fit inside a context window before
the oldest content gets dropped?

**Why this matters:** a context window is the maximum number of tokens a single request
may contain (input + output combined). A model "forgetting" earlier turns is almost
always the caller truncating/dropping older turns to fit — not the model's own memory
fading.

**What we did:** for two realistic windows and two realistic message sizes, computed how
many full user+assistant turns fit.

In [2]:
def turns_until_exhaustion(window, user_tok, assistant_tok, system_tok=0):
    per_turn = user_tok + assistant_tok
    usable = window - system_tok
    full_turns = usable // per_turn
    return full_turns, usable - full_turns * per_turn

cases = [
    ("llama3.2:3b (4K, short msgs)", 4096, 50, 150, 200),
    ("deepseek-r1:8b (8K, longer msgs)", 8192, 120, 400, 500),
]
print(f"{'case':<28}{'window':>7}{'per-turn':>9}{'full_turns':>11}{'left':>7}")
for name, w, u, a, s in cases:
    t, rem = turns_until_exhaustion(w, u, a, s)
    print(f"{name:<28}{w:>7}{u+a:>9}{t:>11}{rem:>7}")

case                         window per-turn full_turns   left
llama3.2:3b (4K, short msgs)   4096      200         19     96
deepseek-r1:8b (8K, longer msgs)   8192      520         14    412


**What this tells us (Finding A2):**
- At the default 4K window with short messages: **19 full turns** fit before the oldest
  turns are dropped.
- At an 8K window with longer messages: only **14 full turns** fit.
- **Implication:** "the model forgot what I said earlier" is a window-arithmetic problem,
  not a memory problem. Long context surviving requires the caller to manage what goes
  in — summarization, sliding windows, or retrieval.

**Source:** `part_a/context_window_math.py` · **Evidence:** `evidence/partA2_context_window.txt`

## A3. Quantization — how a multi-billion-parameter model fits on a laptop

**Question:** why can a 8-billion-parameter model run in RAM that could never hold its
full-precision weights?

**The plain-terms answer:**
- A model's weights are normally stored as 16-bit or 32-bit floating-point numbers.
  At full precision, even a modest model needs enormous RAM: 8.2B params × 16 bits ≈
  **16 GB** just for weights — more than most laptops have available for a single
  process.
- **Quantization** reduces the numeric precision of weights *after* training (commonly
  to 8-bit or 4-bit), trading a small, usually acceptable amount of output quality for a
  large reduction in memory footprint and faster inference.
- **GGUF** is the standard file format the local-model ecosystem (Ollama, LM Studio,
  llama.cpp) uses to package a quantized model. When you `ollama pull` a model today,
  you are pulling a specific quantized GGUF variant, not the research checkpoint. A tag
  like `Q4_K_M` names the exact scheme: **Q4** = 4-bit quantization, **K** = k-means
  scheme, **M** = medium size.

**Our two models, both Q4_K_M (sizes from the live `ollama/tags` call in Part C):**
| Model | Params | Disk size | 16-bit (unquantized) would need | Saved by Q4 |
|---|---|---|---|---|
| llama3.2:3b | 3.2B | 2.0 GB | ~6.4 GB | ~4.4 GB |
| deepseek-r1:8b | 8.2B | 5.2 GB | ~16.4 GB | ~11.2 GB |

**Implication:** modern laptops (16+ GB RAM) consume quantized 7–8B models comfortably
— that is why "run locally" is realistic at all.

# Part B — Prompting Techniques

**Context:** Part A built the mental model (tokens, context, no memory). Part B takes
the five core prompting techniques and applies each one to the **same task** so the
comparison is apples-to-apples:

> **Task:** given a loan applicant description, classify the loan as
> `low_risk` / `medium_risk` / `high_risk`, and answer in strict JSON:
> `{"risk": "<level>", "reason": "<one short sentence>"}`

We then test the interesting comparisons against the real local models in Part C.
The target applicant (same for every variant):
> A civil engineer with 6 years tenure at the same firm, no defaults on record, monthly
> income 3.4x the requested loan installment.

## B1. Zero-Shot vs Few-Shot — showing, not just telling

**Zero-shot** asks for the task with no examples. **Few-shot** shows 3 worked
input→output examples first; well-known best practice is 3–5 diverse, well-formed
examples when format or tone matters, because the model pattern-matches against what
you show it far more reliably than it infers from an instruction alone. Tradeoff: every
example is tokens spent on every request.

In [3]:
TASK = (
    "Read the applicant description and classify the loan as 'low_risk', 'medium_risk', "
    "or 'high_risk'. Answer in a fixed JSON object exactly:\n"
    '{"risk": "<level>", "reason": "<one short sentence>"}'
)
APPLICANT = (
    "Applicant: A civil engineer with 6 years tenure at the same firm, no defaults on "
    "record, monthly income 3.4x the requested loan installment."
)

ZERO_SHOT = TASK + "\n\n" + APPLICANT + "\n\nAnswer:"

FEW_SHOT = (
    "Classify each applicant the same way. Use this exact format:\n"
    '{"risk": "<low_risk|medium_risk|high_risk>", "reason": "<one short sentence>"}\n\n'
    'Example 1:\nApplicant: Freelance graphic designer, income varies month to month, '
    'has one 60-day-late payment last year.\n'
    'Answer: {"risk": "high_risk", "reason": "variable income and a recent late payment '
    'signal repayment unpredictability"}\n\n'
    'Example 2:\nApplicant: Teacher with 5 years at a stable school, clean credit file, '
    'borrowing one-third of annual salary.\n'
    'Answer: {"risk": "low_risk", "reason": "stable employer, clean record, and a small '
    'loan relative to income"}\n\n'
    'Example 3:\nApplicant: Recent college graduate in first job, stable salary, small '
    'existing credit card debt, no defaults.\n'
    'Answer: {"risk": "medium_risk", "reason": "stable-but-short work history with '
    'modest existing debt"}\n\n'
    + APPLICANT
    + "\nAnswer:"
)

print("=== ZERO-SHOT ===")
print(ZERO_SHOT)
print("\n=== FEW-SHOT (3 worked examples) ===")
print(FEW_SHOT)

=== ZERO-SHOT ===
Read the applicant description and classify the loan as 'low_risk', 'medium_risk', or 'high_risk'. Answer in a fixed JSON object exactly:
{"risk": "<level>", "reason": "<one short sentence>"}

Applicant: A civil engineer with 6 years tenure at the same firm, no defaults on record, monthly income 3.4x the requested loan installment.

Answer:

=== FEW-SHOT (3 worked examples) ===
Classify each applicant the same way. Use this exact format:
{"risk": "<low_risk|medium_risk|high_risk>", "reason": "<one short sentence>"}

Example 1:
Applicant: Freelance graphic designer, income varies month to month, has one 60-day-late payment last year.
Answer: {"risk": "high_risk", "reason": "variable income and a recent late payment signal repayment unpredictability"}

Example 2:
Applicant: Teacher with 5 years at a stable school, clean credit file, borrowing one-third of annual salary.
Answer: {"risk": "low_risk", "reason": "stable employer, clean record, and a small loan relative to i

**Files:** `part_b/prompts_task3.py` contains the same prompts — tested live in Part C (T1).

## B2. Chain-of-Thought — asking for the reasoning, not just the answer

Explicitly asking a model to think step by step measurably improves accuracy on
multi-step tasks: the model's own intermediate tokens act as working memory it can
condition on. We wrote two versions of a real reasoning task (compare two loan offers)
— one that requests only the answer, one that requests explicit intermediate math.

In [4]:
NO_COT = (
    "A bank offers a loan of $10,000 at a flat annual interest rate of 6% for 2 years, "
    "to be repaid in 24 equal monthly installments. A second offer is $10,000 at 4.5% for "
    "3 years. Which offer has the LOWER total amount repaid? Show only the final answer "
    "(name the lower offer)."
)

WITH_COT = (
    "A bank offers a loan of $10,000 at a flat annual interest rate of 6% for 2 years, "
    "to be repaid in 24 equal monthly installments. A second offer is $10,000 at 4.5% for "
    "3 years. Which offer has the LOWER total amount repaid? Think step by step: "
    "1) compute total interest of offer 1, 2) add principal for offer 1 total, "
    "3) compute total interest of offer 2, 4) add principal for offer 2 total, "
    "5) compare the two totals and state which is lower."
)

print("=== WITHOUT CoT (answer only) ===")
print(NO_COT)
print()
print("=== WITH CoT (step-by-step requested) ===")
print(WITH_COT)

=== WITHOUT CoT (answer only) ===
A bank offers a loan of $10,000 at a flat annual interest rate of 6% for 2 years, to be repaid in 24 equal monthly installments. A second offer is $10,000 at 4.5% for 3 years. Which offer has the LOWER total amount repaid? Show only the final answer (name the lower offer).

=== WITH CoT (step-by-step requested) ===
A bank offers a loan of $10,000 at a flat annual interest rate of 6% for 2 years, to be repaid in 24 equal monthly installments. A second offer is $10,000 at 4.5% for 3 years. Which offer has the LOWER total amount repaid? Think step by step: 1) compute total interest of offer 1, 2) add principal for offer 1 total, 3) compute total interest of offer 2, 4) add principal for offer 2 total, 5) compare the two totals and state which is lower.


**Reference answer:** Offer 1 = 6% × 2 yr = $1,200 interest → total $11,200.
Offer 2 = 4.5% × 3 yr = $1,350 interest → total $11,350. **Offer 1 is lower.**

**Files:** `part_b/prompts_task4.py` — tested live in Part C (T2, T3).

## B3. Role Prompting — framing

A system prompt or explicit persona shapes tone and priorities by conditioning which
patterns in the model's training are most relevant. We re-framed the loan task with a
domain persona.

In [5]:
ROLE_PROMPT = '''You are a senior credit risk analyst at a Tier-1 bank with 20 years
of underwriting experience. Your judgment determines whether the bank accepts or rejects
a loan application, so you are careful, precise, and evidence-based in every assessment.
You calibrate your risk levels to real banking practice.

Read the applicant description and classify the loan as 'low_risk', 'medium_risk', or
'high_risk'. Answer in a fixed JSON object exactly:
{"risk": "<level>", "reason": "<one short sentence>"}

Applicant: A civil engineer with 6 years tenure at the same firm, no defaults on record,
monthly income 3.4x the requested loan installment.

Answer:'''

print(ROLE_PROMPT)

You are a senior credit risk analyst at a Tier-1 bank with 20 years
of underwriting experience. Your judgment determines whether the bank accepts or rejects
a loan application, so you are careful, precise, and evidence-based in every assessment.
You calibrate your risk levels to real banking practice.

Read the applicant description and classify the loan as 'low_risk', 'medium_risk', or
'high_risk'. Answer in a fixed JSON object exactly:
{"risk": "<level>", "reason": "<one short sentence>"}

Applicant: A civil engineer with 6 years tenure at the same firm, no defaults on record,
monthly income 3.4x the requested loan installment.

Answer:


**Why this persona (and "why not X"):** we chose "senior credit risk analyst at a
Tier-1 bank" because it is domain-grounded and realistic for a loan-underwriting
payload, and it meaningfully conditions the model's calibration. A generic "helpful
assistant" persona was rejected: it adds no conditioning value beyond the task
instruction itself.

**File:** `part_b/prompts_task_role.py` — tested live in Part C (T4).

## B4. Prompt Chaining — breaking a hard task into steps

Prompt chaining splits one complex task into a sequence of smaller prompts, each
output feeding the next — the same "single responsibility" instinct as small functions.
We split the loan task into (1) extract-only and (2) classify-from-facts.

In [6]:
STEP1_EXTRACT = '''You are a data extraction assistant. Given an applicant description,
list ONLY the objective facts as bullet points. Do not comment, do not judge, do not
recommend anything.

Applicant: A civil engineer with 6 years tenure at the same firm, no defaults on record,
monthly income 3.4x the requested loan installment.

Facts:'''

STEP2_CLASSIFY_BASE = '''You are a credit risk classifier. Based solely on the facts below,
classify the loan as 'low_risk', 'medium_risk', or 'high_risk'. Answer in a fixed JSON
object exactly:
{"risk": "<level>", "reason": "<one short sentence>"}

Facts:
{facts}

Answer:'''

print("=== STEP 1: extract facts (no judgment) ===")
print(STEP1_EXTRACT)
print()
print("=== STEP 2: classify from facts (template) ===")
print(STEP2_CLASSIFY_BASE.replace("{facts}", "[step 1 output injected here]"))

=== STEP 1: extract facts (no judgment) ===
You are a data extraction assistant. Given an applicant description,
list ONLY the objective facts as bullet points. Do not comment, do not judge, do not
recommend anything.

Applicant: A civil engineer with 6 years tenure at the same firm, no defaults on record,
monthly income 3.4x the requested loan installment.

Facts:

=== STEP 2: classify from facts (template) ===
You are a credit risk classifier. Based solely on the facts below,
classify the loan as 'low_risk', 'medium_risk', or 'high_risk'. Answer in a fixed JSON
object exactly:
{"risk": "<level>", "reason": "<one short sentence>"}

Facts:
[step 1 output injected here]

Answer:


**File:** `part_b/prompts_task_chain.py` — tested live in Part C (T5).

## B5. Generation Parameters — temperature, top-p, streaming

**Question:** how do you choose settings for a deterministic task versus a creative one?

- **temperature** controls randomness in next-token selection: near 0 ≈ deterministic
  and repeatable; higher values = more varied output at the cost of consistency.
- **top-p** (nucleus sampling) restricts sampling to the smallest set of candidates
  whose cumulative probability exceeds *p*. Most applications tune one or the other,
  not both at once.
- **streaming** returns tokens as generated instead of after completion — needed for a
  responsive UI and to avoid timeouts on long outputs.

Our choice: **temperature 0.1** for every evidence run in this notebook, because every
task here is deterministic (formatting, classification, arithmetic) — we want
repeatable output, not creative variety. The cell below demonstrates the effect of
temperature on the *same* prompt.

In [7]:
import urllib.request, json

prompt = "List 3 reasons why a bank might reject a loan application. Number them 1-3."
for temp in (0.0, 0.8, 1.5):
    body = {"model": "llama3.2:3b", "prompt": prompt, "stream": False,
            "options": {"temperature": temp}}
    req = urllib.request.Request("http://localhost:11434/api/generate",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        r = json.load(resp)
    print(f"===== temperature={temp} =====")
    print(r["response"])
    print()

===== temperature=0.0 =====
Here are 3 reasons why a bank might reject a loan application:

1. **Insufficient Credit History or Poor Credit Score**: If the applicant has a limited or poor credit history, the bank may be hesitant to lend to them. This could be due to a lack of payment history, high credit utilization, or a history of late payments.

2. **Inadequate Collateral or High Risk of Default**: If the applicant is requesting a large loan amount or is using collateral that is not sufficient to secure the loan, the bank may be concerned about the risk of default. This could be due to a lack of assets to recover the loan amount in the event of default.

3. **Inconsistent or Unverifiable Income or Expenses**: If the applicant's income or expenses are not consistent or can't be verified, the bank may be uncertain about the applicant's ability to repay the loan. This could be due to a lack of documentation, inconsistent employment history, or unverifiable income sources.

===== temper

**What this tells us (Finding B5):** the same prompt at temperature 0.0 is
deterministic (same structure, muted style); at 0.8 it stays coherent but wording and
detail shift; at 1.5 coherence begins to break down. For a task where the answer must
be repeatable and correct, low temperature (<0.3) is the right choice; for open-ended
creative output, higher values are appropriate. We logged `temperature: 0.1` on every
evidence run so the results are reproducible.

**Evidence file:** `evidence/part_temperature_comparison.txt`

# Part C — Deploy and Verify Two Real Local Models

## C0. Ollama — install and verify

**Question:** is the runtime installed and running?

In [8]:
!ollama --version
!curl -s http://localhost:11434/api/version

]11;?\ollama version is 0.33.3
{"version":"0.33.3"}

## C1. Model inventory

**What we did:** pulled two open-weight models and listed what the runtime knows about
them — including the exact quantization tags (Q4_K_M) and parameter sizes.

In [9]:
!curl -s http://localhost:11434/api/tags

{"models":[{"name":"deepseek-r1:8b","model":"deepseek-r1:8b","modified_at":"2026-09-15T19:37:50.379729022+05:00","size":5225376047,"digest":"6995872bfe4c521a67b32da386cd21d5c6e819b6e0d62f79f64ec83be99f5763","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"8.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":4096},"capabilities":["completion","thinking"]},{"name":"llama3.2:3b","model":"llama3.2:3b","modified_at":"2026-09-15T19:37:27.715484915+05:00","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"3.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3072},"capabilities":["completion","tools"]}]}

**What this tells us:** both models are Q4_K_M (4-bit quantized) — exactly the
quantization story from A3. llama3.2:3b (`3.2B`, 2.0 GB) and deepseek-r1:8b (`8.2B`,
5.2 GB, flags `thinking`) are real files on disk in
`/var/snap/ollama/common/models/blobs` (6.8 GB total).

## C2. llama3.2:3b — a Llama-family model, verified working

**Question:** does the small Llama model produce a real, coherent, *correct* answer?

In [10]:
import urllib.request, json, time

q = ("What are the 3 most important things a bank should check before approving "
     "a personal loan? Answer concisely in 3 numbered points.")
body = {"model": "llama3.2:3b", "prompt": q, "stream": False, "options": {"temperature": 0.1}}
req = urllib.request.Request("http://localhost:11434/api/generate",
                             data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json"})
t0 = time.time()
with urllib.request.urlopen(req, timeout=300) as resp:
    r = json.load(resp)
dt = time.time() - t0
print(r["response"])
print()
print(f"eval_count: {r['eval_count']}  eval time: {round(r['eval_duration']/1e9, 2)} s  wall: {round(dt, 1)} s")

Here are the 3 most important things a bank should check before approving a personal loan:

1. **Creditworthiness**: Verify the borrower's credit score, credit history, and debt-to-income ratio to assess their ability to repay the loan.
2. **Income and Employment Stability**: Confirm the borrower's income, employment status, and job security to ensure they can afford the loan repayments.
3. **Debt-to-Income Ratio**: Calculate the borrower's debt-to-income ratio to ensure they are not over-extending themselves and can manage their existing debt obligations.

eval_count: 115  eval time: 10.69 s  wall: 11.5 s


**What this tells us (Finding C2):** a real, coherent, correct answer — bank risk is
judged by creditworthiness, income/employment stability, and debt-to-income ratio.
This is the working-verification step, not a formality: the model demonstrably runs
locally and produces useful output at ~10.5 tokens/sec on CPU (tolerable for
interactive chat).

**Evidence file:** `evidence/partC_llama3_real_question.txt`

## C3. deepseek-r1:8b — a current Chinese open-weight model, verified working

**Question:** does the reasoning model produce real output — and do we see its
*unprompted* reasoning first?

In [11]:
import urllib.request, json, time

q = ("What are the 3 most important things a bank should check before approving "
     "a personal loan? Answer concisely in 3 numbered points.")
body = {"model": "deepseek-r1:8b", "prompt": q, "stream": False,
        "think": True, "options": {"temperature": 0.1}}
req = urllib.request.Request("http://localhost:11434/api/generate",
                             data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json"})
t0 = time.time()
with urllib.request.urlopen(req, timeout=600) as resp:
    r = json.load(resp)
print("=== THINKING (unprompted, first 1200 chars) ===")
print(r.get("thinking", "")[:1200])
print("...")
print()
print("=== RESPONSE ===")
print(r["response"])
print()
print(f"eval_count: {r['eval_count']}  eval time: {round(r['eval_duration']/1e9, 2)} s  wall: {round(time.time()-t0, 1)} s")

=== THINKING (unprompted, first 1200 chars) ===
The user asked for the three most important things a bank should check before approving a personal loan, and they want it answered concisely in three numbered points. I need to make sure my response is direct and to the point, avoiding any fluff.

First, I should think about what banks prioritize when assessing loan applications. Personal loans are risky for lenders, so they focus on minimizing defaults. The key factors are usually related to the borrower's ability to repay and their financial stability.

The most critical thing is the borrower's creditworthiness. That includes checking credit history, score, and reports for any red flags like late payments or bankruptcies. A good credit score indicates reliability, so this is a top check.

Next, the bank needs to evaluate the borrower's repayment ability. This involves looking at income sources, employment stability, and existing debts. They calculate debt-to-income ratio to ensure the l

**What this tells us (Finding C3):** deepseek-r1:8b outputs its own visible
step-by-step reasoning **before** the final answer, *without being asked* — reasoning
is built into how this model was trained (it is listed as a `thinking`-capable model).
Its answer is coherent and correct.

**Evidence file:** `evidence/partC_deepseek_r1_thinking.txt`

# Part C → Live Prompt Testing against the Real Models

## T1. Zero-shot vs Few-shot on llama3.2:3b (Kata #8)

**Question:** did the few-shot version actually produce a more reliably-formatted
answer on this *local* model?

In [12]:
import urllib.request, json

def ask(prompt):
    body = {"model": "llama3.2:3b", "prompt": prompt, "stream": False,
            "options": {"temperature": 0.1}}
    req = urllib.request.Request("http://localhost:11434/api/generate",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.load(resp)["response"]

print("=== ZERO-SHOT ===")
print(ask(ZERO_SHOT))
print()
print("=== FEW-SHOT ===")
print(ask(FEW_SHOT))

=== ZERO-SHOT ===
{"risk": "low_risk", "reason": "stable income and no credit history issues"}

=== FEW-SHOT ===
{"risk": "low_risk", "reason": "stable employment, clean record, and a high income relative to loan installment"}


**Finding T1 (honest note):** both variants produced valid JSON and the same
classification (`low_risk`). The few-shot reason is slightly more specific — it cites
employment stability + higher income relative to the installment — whereas zero-shot
says only "stable income and no credit history issues". **Honestly: on this local 3B
model, zero-shot already handled the constrained JSON format correctly; the few-shot
advantage here was modest (slightly richer justification), not a difference in format
validity.** Local models don't always behave identically to hosted frontier models on
the same technique.

**Evidence file:** `evidence/partC_zeroshot_vs_fewshot_llama.txt`

## T2. Chain-of-Thought on llama3.2:3b (part of Kata #9)

**Question:** what does asking for step-by-step reasoning change on the smaller model?

In [13]:
def ask(prompt):
    body = {"model": "llama3.2:3b", "prompt": prompt, "stream": False,
            "options": {"temperature": 0.1}}
    req = urllib.request.Request("http://localhost:11434/api/generate",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.load(resp)["response"]

print("=== WITHOUT CoT ===")
print(ask(NO_COT))
print()
print("=== WITH CoT ===")
print(ask(WITH_COT))

=== WITHOUT CoT ===
The second offer ($10,000 at 4.5% for 3 years) has the lower total amount repaid.

=== WITH CoT ===
To determine which offer has the lower total amount repaid, let's calculate the total interest and principal for each offer.

**Offer 1: 6% for 2 years**

1. Calculate total interest:
   Principal = $10,000
   Interest Rate = 6% = 0.06
   Time = 2 years = 24 months
   Total Interest = Principal x Rate x Time
   Total Interest = $10,000 x 0.06 x 24
   Total Interest = $14,400

2. Calculate total amount repaid:
   Total Amount = Principal + Total Interest
   Total Amount = $10,000 + $14,400
   Total Amount = $24,400

**Offer 2: 4.5% for 3 years**

1. Calculate total interest:
   Principal = $10,000
   Interest Rate = 4.5% = 0.045
   Time = 3 years = 36 months
   Total Interest = Principal x Rate x Time
   Total Interest = $10,000 x 0.045 x 36
   Total Interest = $16,200

2. Calculate total amount repaid:
   Total Amount = Principal + Total Interest
   Total Amount = $10

**Finding T2:** without CoT the model simply named the wrong offer (no reasoning to audit).
With CoT the model produced visible steps — but its flat-rate formula was wrong
(it used 24 months where it should use 2 years: $10,000 × 0.06 × 24 = $14,400). So
"asked reason" makes the reasoning *auditable*, and can visibly help, but it does not
guarantee the math is right.

## T3. Chain-of-Thought vs DeepSeek-R1's unprompted reasoning (Kata #9, core finding)

**Question:** what is the actual difference between "a model that reasons because you
asked it to" and "a model that reasons because that's how it was trained to respond"?
We sent the *same* no-CoT prompt (B2) to R1 with `think: True`. R1 has no bounded
answering behavior: on this puzzle it did not settle within 10+ minutes. We therefore
quote the captured run here (exploratory phase, saved to evidence) rather than rerun an
unbounded generation inside the notebook — a notebook that must complete is a bounded
experiment.

In [14]:
import re

# Bounded: read the already-captured run instead of re-running an unbounded generation
try:
    text = open("evidence/partC_r1_nocot_full.txt", encoding="utf-8").read()
except FileNotFoundError:
    text = "evidence/partC_r1_nocot_full.txt not found; see part9_cot_contrast_analysis.txt"
print("Showing the captured R1 run from the exploratory phase (see"
      "\nevidence/partC_r1_nocot_full.txt for the full text):\n---")
print(text[:1400])
print("---")
print()
print("Two-sentence contrast (also in evidence/part9_cot_contrast_analysis.txt):")
print("> A model that reasons because you asked it to (Llama CoT) steps through the")
print("> requested structure and settles on an answer — auditable, because it printed")
print("> the steps, but still wrong here (it applied months instead of years to the")
print("> flat-rate formula).  A model that reasons because it was trained to reason")
print("> (DeepSeek-R1) goes deeper unprompted — it correctly detected the ambiguity of")
print("> 'flat annual rate' in this puzzle — but that same depth can turn into")
print("> over-analysis: here R1 never reached a final answer within 10+ minutes.")

Showing the captured R1 run from the exploratory phase (see
evidence/partC_r1_nocot_full.txt for the full text):
---
=== THINKING (first 500 chars) ===
I need to compare two loan offers and figure out which one has the lower total amount repaid. The first offer is $10,000 at a flat annual interest rate of 6% for 2 years, repaid in 24 equal monthly installments. The second offer is $10,000 at 4.5% for 3 years. I need to find out which one costs less in total.

First, I should understand what a "flat annual interest rate" means. I think it might be different from the standard annual percentage rate (APR) because it says "flat," which could imply 
...

=== THINKING (last 500 chars) ===
ed on the initial principal for the entire loan period, and then the total interest is added to the principal, and divided by the number of payments.

For example, in some car loans or something.

But for the second loan, it's not specified.

Perhaps the second loan is a simple interest loan for 3 years at 

**Finding T3 (the two-sentence core insight):**
> "A model that reasons because you asked it to (Llama CoT) follows the requested step
> structure mechanically and reaches an answer — but the math can still be wrong, which
> is why the shown reasoning is so valuable: you can audit it.
> A model that reasons because that's how it was trained to respond (DeepSeek-R1)
> produces its own unprompted reasoning, and it goes *deeper* — here it detected the
> genuine ambiguity in 'flat annual rate' for the second offer, which the simpler model
> missed — but the same depth can turn into over-analysis: R1 got stuck in a loop and
> never reached a final answer on this arithmetic puzzle within the allowed time."

**R1 never settling is itself a finding (honest):** it is a documented limitation of
model+task, not something to hide. Its early reasoning was correct ($1,200 for Offer 1);
it then spiraled on the ambiguity instead of concluding. Evidence captures the loop.

**Evidence files:** `evidence/partC_cot_comparison_llama.txt`, `evidence/partC_r1_nocot_full.txt`, `evidence/part9_cot_contrast_analysis.txt`

## T4. Role Prompting — live on both models

**Question:** does the persona change the answer vs the plain zero-shot version?

In [15]:
import urllib.request, json

for model in ("llama3.2:3b", "deepseek-r1:8b"):
    body = {"model": model, "prompt": ROLE_PROMPT, "stream": False,
            "options": {"temperature": 0.1}}
    req = urllib.request.Request("http://localhost:11434/api/generate",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as resp:
        r = json.load(resp)
    print(f"===== {model} =====")
    if r.get("thinking"):
        print("THINKING (first 200 chars):", r["thinking"][:200], "...")
    print(r["response"])
    print()

===== llama3.2:3b =====
{"risk": "low_risk", "reason": "stable income and no credit history issues"}

===== deepseek-r1:8b =====
THINKING (first 200 chars): Hmm, the user wants me to act as a senior credit risk analyst at a Tier-1 bank with 20 years of experience. They need me to classify a loan application as low, medium, or high risk based on the applic ...
```json
{"risk": "medium_risk", "reason": "The applicant has stable employment and no defaults, but the relatively short tenure (6 years) and moderate income multiple (3.4x) suggest a slightly elevated risk profile compared to highly tenured, income-rich borrowers."}
```



**Finding T4:** the persona produces richer, more calibrated justifications than the
plain zero-shot prompt, and it keeps the JSON format valid. Important honesty check: a
model is *stateful even at temperature 0.1* — 0.1 reduces variance but does not remove
it. Llama returned `low_risk` in every run; deepseek-r1:8b returned `low_risk` in one
run and `medium_risk` in a re-run, each with a defensible domain reason (stable job/no
defaults vs. only 6 years of tenure). The demonstration the spec asks for — a persona
instruction shapes tone, calibration, and priority — holds in both cases; the exact
classification is not bit-fixed across runs.

**Evidence file:** `evidence/part_role_prompting.txt`


## T5. Prompt Chaining — live on llama3.2:3b

**Question:** does splitting extract-then-classify change the output vs asking directly?

In [16]:
def ask(prompt):
    body = {"model": "llama3.2:3b", "prompt": prompt, "stream": False,
            "options": {"temperature": 0.1}}
    req = urllib.request.Request("http://localhost:11434/api/generate",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        return json.load(resp)["response"]

# Step 1: extract facts only
facts = ask(STEP1_EXTRACT)
print("=== STEP 1 OUTPUT (facts only) ===")
print(facts)
print()

# Step 2: classify from facts
step2 = STEP2_CLASSIFY_BASE.replace("{facts}", facts)
print("=== STEP 2 INPUT ===")
print(step2)
print()
print("=== STEP 2 OUTPUT ===")
print(ask(step2))

=== STEP 1 OUTPUT (facts only) ===
• The applicant is a civil engineer.
• The applicant has 6 years of tenure at the same firm.
• The applicant has no defaults on record.
• The applicant's monthly income is 3.4 times the requested loan installment.

=== STEP 2 INPUT ===
You are a credit risk classifier. Based solely on the facts below,
classify the loan as 'low_risk', 'medium_risk', or 'high_risk'. Answer in a fixed JSON
object exactly:
{"risk": "<level>", "reason": "<one short sentence>"}

Facts:
• The applicant is a civil engineer.
• The applicant has 6 years of tenure at the same firm.
• The applicant has no defaults on record.
• The applicant's monthly income is 3.4 times the requested loan installment.

Answer:

=== STEP 2 OUTPUT ===
{"risk": "low_risk", "reason": "Strong employment history and stable income"}


**Finding T5:** the chain cleanly separates concerns: Step 1 returned pure bulleted
facts (no judgment leaked), Step 2 consumed only those facts and returned a clean,
valid JSON classification. Compared to the single-shot versions, chaining made the
process auditable (you can check what facts the classifier saw) and held the format —
at the cost of two calls / more tokens. This is the same instinct as small functions
vs one giant function, applied to prompts.

**Evidence file:** `evidence/part_prompt_chaining.txt`

# Synthesis — What We Learned Today

## The one idea that ties the whole day together
A language model is a **real file on disk** that a program loads and runs, generating
one token at a time, with **no memory outside the current request**. Everything else —
tokens, context windows, prompting techniques, quantization — is engineering around
that single fact.

## Prompting techniques — when to use what

| Technique | What it does | When to use | Cost |
|---|---|---|---|
| Zero-shot | Task instruction only | Simple tasks, known format | Lowest tokens |
| Few-shot | 3–5 worked examples | Format/tone matters | +tokens per example |
| Chain-of-thought | Ask for the reasoning | Multi-step reasoning | +tokens, more auditable |
| Role prompting | Persona/system prompt | Calibrate tone & priority | Cheap (fixed) |
| Prompt chaining | Split into single-purpose calls | Complex multi-part tasks | Multiple calls |

## Resource usage (Kata #10, first impressions)
- **Disk:** 6.8 GB total for both Q4_K_M models (2.0 GB llama3.2:3b + 5.2 GB r1:8b).
- **Speed on CPU:** llama3.2:3b ≈ 10.5 tokens/s (interactive-comfortable);
  deepseek-r1:8b ≈ 3.9 tokens/s, and with thinking enabled a single hard query can take
  many minutes (on the ambiguous loan puzzle, R1 never settled within 10 min).
- **RAM/swap:** both models loaded cleanly into 31 GiB RAM; no swap, no crash.

**Evidence:** `evidence/part10_resource_usage.txt`

## Local vs hosted (from today's framing, to be tested tomorrow)
Local today: data never leaves the machine, no per-token cost, no rate limits,
no network dependency — but needs local compute, weaker capability than the largest
frontier models, and the ops burden is yours. Tomorrow we compare against a hosted API.

## Honest limitations — what we could NOT do
- **Correctness is not guaranteed:** local 3B/8B models can give wrong answers (the
  CoT loan math) while sounding plausible. We verified real output but not perfect
  accuracy for every task.
- **One-shot comparisons:** prompt comparisons were run once each (temperature 0.1),
  not replicated. Sampling-variance estimates were not computed.
- **No GPU offload measured:** the machine has an Intel UHD 620; Ollama ran on CPU.
  Token/s numbers are CPU-only and would improve with a discrete GPU.
- **The ambiguous loan puzzle (R1, T3) was not fully resolved** by the model within
  the time budget — a limitation of the model on that input, which we documented
  rather than hiding.
- **Context-window arithmetic uses estimates of per-message tokens**, not measured
  per-conversation token counts.

# Reproducibility & Determinism Notes

- **Environment:** Python 3.10, Ollama 0.33.3 (snap), `tiktoken==0.14.0`,
  `jupyter`, `nbformat` (`requirements.txt`).
- **Determinism vs the EDA project:** model generation is *not* deterministic even at
  temperature 0.1 (stateful sampling); unlike the byte-identical artifacts of the ML
  Pipeline project, this notebook's model outputs can vary slightly run-to-run.
  Demonstrated live in this notebook: llama returned `low_risk` on the role-prompt
  task across runs, while R1 returned `low_risk` once and `medium_risk` on a re-run
  (Finding T4). This is expected, not a bug.
- **What we can reproduce exactly:** tokenizer math (42 words → 51 tokens), context
  window arithmetic (19/14 turns), all API parameters (model name, quantization,
  temperature 0.1) are fixed and logged on every evidence file.
- **What varies run-to-run:** the model's generated text/classification. The
  qualitative findings in this report (persona shapes tone, CoT makes reasoning
  auditable, R1 goes deeper unprompted) hold across runs; exact wording and exact
  risk labels do not.
- **Fresh-run proof:** this notebook was executed top-to-bottom (Restart & Run All)
  with zero error cells; see results embedded above.
